# 🧪 Test Notebook: End-to-End Integration Pipeline with Schema Mapper
Kiểm thử luồng LangGraph hoàn chỉnh từ `query_parser` -> `data_discovery` -> `schema_mapper` -> `code_generator` -> `executor`

*Toàn bộ kết quả log được ghi tự động ra file `test_pipeline_integration_log.txt`.*

In [ ]:
import os
import sys
import io
import json
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# ==============================================================================
# Dual Output Logger: Ghi toàn bộ kết quả in ra cả Console và file .txt
# ==============================================================================
class DualOutputLogger:
    """Redirects sys.stdout so that all outputs are printed to console/notebook AND saved into a .txt log file."""
    def __init__(self, log_filepath: str = "pipeline_execution.txt", stream=None):
        if stream is not None:
            self.terminal = stream
        elif isinstance(sys.stdout, DualOutputLogger):
            self.terminal = sys.stdout.terminal
        else:
            self.terminal = sys.stdout if sys.stdout is not None else sys.__stdout__
        self.log_filepath = Path(log_filepath)
        self.log_filepath.parent.mkdir(parents=True, exist_ok=True)
        self.log_file = open(self.log_filepath, "a", encoding="utf-8", errors="replace")

    def write(self, message):
        try:
            if hasattr(self.terminal, "write"):
                self.terminal.write(message)
        except Exception:
            pass
        try:
            if hasattr(self, "log_file") and not self.log_file.closed:
                self.log_file.write(message)
                self.log_file.flush()
        except Exception:
            pass

    def flush(self):
        try:
            if hasattr(self.terminal, "flush"):
                self.terminal.flush()
        except Exception:
            pass
        try:
            if hasattr(self, "log_file") and not self.log_file.closed:
                self.log_file.flush()
        except Exception:
            pass

    def isatty(self) -> bool:
        if hasattr(self.terminal, "isatty"):
            try:
                return self.terminal.isatty()
            except Exception:
                return False
        return False

    def fileno(self):
        if hasattr(self.terminal, "fileno"):
            return self.terminal.fileno()
        raise io.UnsupportedOperation("fileno not supported")

    def readable(self) -> bool:
        return False

    def writable(self) -> bool:
        return True

    def seekable(self) -> bool:
        return False

    @property
    def encoding(self):
        return getattr(self.terminal, "encoding", "utf-8")

    @property
    def errors(self):
        return getattr(self.terminal, "errors", "replace")

    def close(self):
        if hasattr(self, "log_file") and not self.log_file.closed:
            self.log_file.close()

    def __getattr__(self, attr):
        return getattr(self.terminal, attr)

LOG_FILE = PROJECT_ROOT / "notebooks" / "test_pipeline_integration_log.txt"
if not isinstance(sys.stdout, DualOutputLogger):
    sys.stdout = DualOutputLogger(str(LOG_FILE))
print(f"📝 Đã kích hoạt ghi log tự động ra file: {LOG_FILE.resolve()}")

from pipeline.src.config import config
from pipeline.src.graph import create_cocopila_graph
from pipeline.src.state import AgentState

print("✅ Đang khởi tạo LangGraph Workflow Graph...")
app = create_cocopila_graph(config)
print("✅ Graph đã biên dịch thành công!")

### 1. Chạy câu hỏi kiểm thử qua toàn bộ Workflow

In [ ]:
test_query = "Tài sản ngắn hạn của VNM vào ngày 31/12/2023 là bao nhiêu?"

initial_state = {
    "user_query": test_query,
    "status": "pending",
    "retry_count": 0,
    "node_latencies": {}
}

print(f"🚀 Bắt đầu thực thi Pipeline với câu hỏi: '{test_query}'\n")
final_state = app.invoke(initial_state)

print("="*60)
print("🏁 KẾT QUẢ CUỐI CÙNG:")
print(f"Status: {final_state.get('status')}")
print(f"Discovered Tables: {len(final_state.get('discovered_tables', []))}")
print(f"Schema Useful Columns: {final_state.get('schema', {}).get('useful_columns')}")
print(f"Schema Sub-sections: {final_state.get('schema', {}).get('sub_sections')}")
print(f"Generated Code:\n{final_state.get('generated_code')}")
print(f"Execution Result: {final_state.get('execution_result')}")
print(f"Latencies: {final_state.get('node_latencies')}")
print("="*60)

### 2. Kiểm tra độ trễ (Latency Benchmark) từng Node

In [ ]:
latencies = final_state.get("node_latencies", {})
for node, sec in latencies.items():
    print(f" - Node {node:18s}: {sec:.3f} s")
print(f"👉 Tổng thời gian: {sum(latencies.values()):.3f} s")
print(f"📝 File log toàn bộ tiến trình: {LOG_FILE.resolve()}")